# Tripwire: retrain the tool-output injection classifiers (Phase 9, Colab / Kaggle T4)

Retrains **MiniLM-L6** (default, base `sentence-transformers/all-MiniLM-L6-v2`, chosen in Phase 4) and **DeBERTa-v3-small** (accuracy reference) on the same dataset and splits (seed 42). New since Phase 4:

* **Realistic tool outputs** (`scripts/build_tool_outputs.py`): HTML pages with analytics, consent and CMS boilerplate, flattened API JSON, docs and notices, about 6,000 benign rows in 36 formats, plus about 1,500 copies with an injection planted in the same format. Whole formats are held out for val and test.
* **Leakage check**: every row sharing a word 8-gram with the gateway's Phase 3 corpus (`gateway/tests/fixtures/`) is dropped; counts go to `data/stats.json`.
* **More human-written attacks**: Lakera Gandalf (MIT) and jackhhao jailbreaks (Apache-2.0); deepset up-weighted x3 in train.
* **DeBERTa int8 fix**: the exporter tries five quantization settings and keeps the first within 0.05 of PyTorch (`config.json` records which).
* **Acceptance gate** (`scripts/gate.py`): Phase 3 corpus classifier-only false positives <= 2% and test_ood recall >= 98%. The notebook only reports it; publishing is decided afterwards.

1. Runtime → Change runtime type → **T4 GPU** (Kaggle: Accelerator → GPU T4). Python 3.12 or newer.
2. **Run all.** Every step stops the notebook with a clear `STEP FAILED` error if it fails. The preflight cell exports a tiny model in seconds, so export problems show up before any training.
3. The last cell checks every artifact against the gateway contract, prints a summary table and writes **`tripwire-artifacts.zip`** (models, configs, metrics, gate result, dataset stats). Download it and send it back.

Rough time on a free T4: dataset about 5 min, MiniLM training about 15 min, DeBERTa-v3-small training about 50 min, int8 export checks about 10 min. Evaluation runs the int8 ONNX files on CPU one window at a time (exactly like the gateway): about 10 min for MiniLM and 45–60 min for DeBERTa. Plan for about 2.5 hours; keep the tab active (free Colab disconnects idle sessions).

Prompt Guard 2 is gated (Llama 4 Community License). Request access on its Hugging Face page and add an `HF_TOKEN` secret (Colab: key icon; Kaggle: Add-ons → Secrets), or set `EVAL_PROMPT_GUARD = False`. Its weights are evaluated only; they are never zipped or published.


In [ ]:
import os, subprocess, sys, json, pathlib

def sh(cmd: str, step: str) -> None:
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"STEP FAILED ({step}): `{cmd}` exited with {p.returncode}. Fix the error above and re-run from this cell.")

def require(paths, step: str) -> None:
    missing = [str(p) for p in paths if not pathlib.Path(p).exists()]
    if missing:
        raise RuntimeError(f"STEP FAILED ({step}): expected output missing: {missing}")

print("Python", sys.version.split()[0])
if sys.version_info < (3, 12):
    raise RuntimeError("STEP FAILED (runtime): Python 3.12+ required (pinned numpy needs it)")
sh("nvidia-smi -L || echo 'no GPU: training will be slow'", "gpu check")
if not pathlib.Path("scripts/preflight.py").exists():  # re-running this cell is safe
    if not pathlib.Path("tripwire-agent-gateway").exists():
        sh("git clone --depth 1 https://github.com/yugpatill/tripwire-agent-gateway.git", "clone")
    os.chdir("tripwire-agent-gateway/ml")
sh("git log -1 --oneline", "clone")

In [ ]:
# 1. Remove preinstalled packages that need huggingface_hub>=1 (our transformers pin needs <1.0);
#    optimum's exporter imports diffusers when it is present and fails.
sh("pip uninstall -y -q diffusers gradio gradio-client", "uninstall conflicting packages")
# 2. Exact pins (same versions as ml/uv.lock). torch is not in the file: the runtime's CUDA build is kept.
sh("pip install -q -r requirements-colab.txt", "install pinned requirements")

In [ ]:
# Fails in seconds if anything is off: Python, torch range, pins, conflicting packages,
# imports, and a tiny BERT + DeBERTa ONNX export, int8 quantization and run.
sh("python scripts/preflight.py", "preflight")

In [ ]:
EVAL_PROMPT_GUARD = True   # zero-shot baseline; needs HF_TOKEN with approved access

try:
    from google.colab import userdata  # Colab
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
if EVAL_PROMPT_GUARD and not os.environ.get("HF_TOKEN"):
    print("Prompt Guard 2 will be skipped (no HF_TOKEN). Set EVAL_PROMPT_GUARD = False to silence this.")

In [ ]:
sh("python scripts/build_dataset.py", "build dataset")
require([f"data/{s}.jsonl" for s in ("train", "val", "test", "test_ood")], "build dataset")
stats = json.loads(pathlib.Path("data/stats.json").read_text())
for split, s in stats.items():
    if split.startswith("_"):
        continue
    if s["_total"] == 0 or s["_injection"] == 0:
        raise RuntimeError(f"STEP FAILED (build dataset): split {split} is empty or has no injections")
print({k: (v["_total"], v["_injection"]) for k, v in stats.items() if not k.startswith("_")})
print("dropped for sharing an 8-gram with the Phase 3 corpus:", stats.get("_leakage_dropped_by_source"))


### MiniLM-L6 base
Phase 4 compared `nreimers/MiniLM-L6-H384-uncased` and `sentence-transformers/all-MiniLM-L6-v2` (one epoch each, validation PR-AUC 0.9951 vs 0.9987) and picked the sentence-transformers checkpoint. The retrain keeps that choice, which saves about 10 minutes.


In [ ]:
best_hf = "sentence-transformers/all-MiniLM-L6-v2"
pathlib.Path("artifacts").mkdir(exist_ok=True)
pathlib.Path("artifacts/minilm_selection.json").write_text(json.dumps({"selected": best_hf, "note": "fixed from the Phase 4 selection"}, indent=2))
print("base:", best_hf)


In [ ]:
def train_export_evaluate(key: str, extra: str = "") -> None:
    art = pathlib.Path(f"artifacts/injection-{key}")
    sh(f"python scripts/train.py --model {key} {extra}", f"train {key}")
    require([f"checkpoints/{key}/best/train_info.json"], f"train {key}")
    sh(f"python scripts/export_onnx.py --model {key}", f"export {key}")
    require([art / "model.int8.onnx", art / "tokenizer.json", art / "config.json"], f"export {key}")
    sh(f"python scripts/evaluate.py --model {key} --threads 2", f"evaluate {key}")
    require([art / "metrics.json"], f"evaluate {key}")

train_export_evaluate("minilm-l6", f"--hf-id {best_hf}")

In [ ]:
train_export_evaluate("deberta-v3-small")

In [ ]:
if EVAL_PROMPT_GUARD and os.environ.get("HF_TOKEN"):
    art = pathlib.Path("artifacts/injection-prompt-guard-2-22m")
    sh("python scripts/export_onnx.py --model prompt-guard-2-22m", "export prompt-guard-2-22m (check that access is approved on Hugging Face)")
    require([art / "model.int8.onnx", art / "config.json"], "export prompt-guard-2-22m")
    sh("python scripts/evaluate.py --model prompt-guard-2-22m --threads 2", "evaluate prompt-guard-2-22m")
    require([art / "metrics.json"], "evaluate prompt-guard-2-22m")
else:
    print("Prompt Guard 2 skipped.")

In [ ]:
# Acceptance gate: report only (the decision to publish is made after you send the zip).
p = subprocess.run("python scripts/gate.py --model minilm-l6 --model deberta-v3-small", shell=True, text=True, capture_output=True)
print(p.stdout, p.stderr)
print("GATE:", "all models pass" if p.returncode == 0 else "at least one model fails (see above); continue to the zip anyway")


In [ ]:
# Contract check (model.int8.onnx, tokenizer.json, config.json {max_length, threshold, labels},
# metrics.json; loads and scores in onnxruntime), summary table, then the zip.
sh("python scripts/check_artifacts.py --require minilm-l6,deberta-v3-small --zip tripwire-artifacts.zip", "artifact contract check")
require(["tripwire-artifacts.zip"], "zip")
try:
    from google.colab import files
    files.download("tripwire-artifacts.zip")
except Exception:
    print("Download tripwire-artifacts.zip from the file browser (Kaggle: /kaggle/working/tripwire-agent-gateway/ml).")